In [1]:
import numpy as np 
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from tqdm import tqdm

## 度量不同脑区的回声物种差异

In [2]:
scdata = sc.read_h5ad('/home/rsun@ZHANGroup.local/sly_data/data/final_anno.h5ad')
scdata

AnnData object with n_obs × n_vars = 338818 × 3000
    obs: 'species', 'tissues', 'samples', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'leiden', 'cluster', 'celltype', 'coarse_celltype'
    var: 'gene_ids', 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'celltype_colors', 'cluster_colors', 'coarse_celltype_colors', 'dendrogram_leiden', 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'samples_colors', 'species_colors', 'tissues_colors', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap', 'harmony_0', 'umap_0'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [ ]:


metric_key = 'euclidean'
tissue_res = {}
for tissue in scdata.obs.tissues.unique():

    res_dic = {}

    adata = scdata[scdata.obs['tissues'] == tissue,:] # tissue spcific data
    for species in ['CY','JT','T','QF','M']:
        species_idx = adata.obs['species'] == species 
        subdata = adata[species_idx,:] 
        meta_cell = subdata.obsm['X_pca_harmony'].mean(axis = 0)
        res_dic[species] = meta_cell 
        
    
    res_df = pd.DataFrame(res_dic).T
    sim_df = squareform(pdist(res_df, metric=metric_key))

    if tissue == 'HM':
        hm_sim = sim_df
    score = 2*(sim_df[:3,3:].mean()) - (sim_df[:3,:3].sum()/6 + sim_df[3:,3:].sum()/2)
    tissue_res[tissue] = score 

In [ ]:
diff = np.array(list(tissue_res.values()))
tmp = np.exp(diff/5)
tmp = tmp/tmp.sum()
plt.bar(x =tissue_res.keys(), height = tmp)
plt.xticks(rotation = 45)
plt.show()